In [1]:
# =============================================================================
# GUS01F: Loading Numerical Data onto GeoTERYT Records
# =============================================================================
# This notebook demonstrates the v4.0 data storage capabilities:
# 1. Load BDL demographic data (subject P2137 - population)
# 2. Process and attach time series to TERYTRecord objects
# 3. Query data on individual records
# 4. Aggregate for regions (voivodeships)
# 5. Produce joint/marginal distributions
# 6. Save/reload database with data persistence
# =============================================================================

# STEP 1: Imports and Path Setup
import os
import sys
from pathlib import Path
import importlib
import gc

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# Find the repository root
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists():
            return p
    return start

repo_root = find_repo_root()
tools_path = repo_root / 'Code' / 'tools'
if str(tools_path) not in sys.path:
    sys.path.insert(0, str(tools_path))

# Reload geoTERYT_db to get latest version (v4.0)
import geoTERYT_db as gtdb
importlib.reload(gtdb)

# Data paths
data_root = repo_root.parent.parent / 'Data'
geo_root = data_root / 'Geospatial'
gus_root = data_root / 'GUS'

print(f"Repository root: {repo_root}")
print(f"Data root: {data_root}")
print(f"GUS root: {gus_root}")

Repository root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper
Data root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data
GUS root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/GUS


In [2]:
# =============================================================================
# STEP 2: Load Complete GeoTERYT Database
# =============================================================================
complete_db_path = geo_root / 'geoteryt_complete_geom_OW.pkl'
db = gtdb.load_complete_database(complete_db_path)
db.print_summary()

Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_geom_OW.pkl...
  Database version: 4.2
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4560 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3612
  ✓ Records with old_woj: 2658
GeoTERYT Database Summary (v3.0)
Total records:           4,560
Year range:              1999 - 2024
------------------------------------------------------------
Administrative levels:
  Voivodeships (2):      16
  Powiats (5):           382
  Gminas (6):            4162
------------------------------------------------------------
Change tracking:
  Records with changes:      772
  Records with level changes: 0
  Records with kind changes:  0
------------------------------------------------------------
Geometry:
  Records with geometr

In [3]:
# =============================================================================
# STEP 2B: Link Children to Parents (must happen before data loading)
# =============================================================================
db.link_children_to_parents()

Linking child units to their parents...
  ✓ Linked children to parents for 4560 records


In [4]:
# =============================================================================
# STEP 2C: Load Pre-1999 Voivodeship Boundaries (from GUS01E)
# =============================================================================
# pre1999_path = geo_root / 'voivodeships_1975_final.geojson'
pre1999_path = geo_root / 'voivodeships_1975_49_clean_EPSG2180.geojson'

if not pre1999_path.exists():
    raise FileNotFoundError(f"Pre-1999 boundaries not found: {pre1999_path}\nRun 02_borders.ipynb first!")

pre1999_gdf = gpd.read_file(pre1999_path)
pre1999_gdf = pre1999_gdf.to_crs('EPSG:2180')  # Ensure projected CRS
pre1999_gdf['teryt_id'] = pre1999_gdf['voiv_id'].apply(lambda x: str(100-x) + "00000")

print(f"Loaded {len(pre1999_gdf)} pre-1999 voivodeships")
print(f"Expected: 49 voivodeships")
print(f"\nVoivodeship areas (sample):")
for idx, row in pre1999_gdf.head(10).iterrows():
    area_km2 = row.geometry.area / 1e6
    print(f"  {row['name']}: {area_km2:,.0f} km²")

# Sanity check: total area should be ~312,685 km²
total_area = pre1999_gdf.geometry.area.sum() / 1e6
print(f"\nTotal area: {total_area:,.0f} km² (Poland ≈ 312,685 km²)")

Loaded 49 pre-1999 voivodeships
Expected: 49 voivodeships

Voivodeship areas (sample):
  bialskopodlaskie: 5,348 km²
  białostockie: 10,069 km²
  bielskie: 3,700 km²
  bydgoskie: 10,344 km²
  chełmskie: 3,879 km²
  ciechanowskie: 6,356 km²
  częstochowskie: 6,176 km²
  elbląskie: 6,110 km²
  gdańskie: 7,388 km²
  gorzowskie: 8,483 km²

Total area: 312,648 km² (Poland ≈ 312,685 km²)


In [5]:
# =============================================================================
# STEP 2D: Create Fake TERYTRecords for Old Voivodeships + Split Mazowieckie
# =============================================================================
from shapely.ops import unary_union

# Store old voivodeships GeoDataFrame on the database (will be saved/loaded)
db.set_old_voivodship_gdf(pre1999_gdf)

# Create fake TERYTRecord for each old voivodeship (pre-1999)
for idx, row in pre1999_gdf.iterrows():
    record = gtdb.TERYTRecord(teryt_id=row['teryt_id'],
                              name=str(row['name']).capitalize())
    record.level = 2   # voivodeship level
    record.kind = 0
    record.geometry = row.geometry
    record.parent_id = '0000000'
    
    # Find children: gminas whose old_woj matches this voivodeship name
    children = []
    for tid, rec in db._records.items():
        if rec.old_woj and str(rec.old_woj).lower() == str(row['name']).lower():
            children.append(tid)
    record.children_ids = children
    
    db._records[record.teryt_id] = record

print(f"Added {len(pre1999_gdf)} old voivodeship records to database")

# --- REGION WARSZAWSKI STOŁECZNY ---
record_WAW = gtdb.TERYTRecord(teryt_id="1300000",
                              name="Warszawski stołeczny")
record_WAW.level = 2
record_WAW.kind = 0
record_WAW.parent_id = '0000000'
record_WAW.children_ids = ['1421000','1418000','1432000','1434000','1417000',
                           '1414000','1405000','1408000','1412000','1431000']
# Build geometry from children
waw_geoms = [db._records[tid].geometry for tid in record_WAW.children_ids
             if tid in db._records and db._records[tid].geometry is not None]
if waw_geoms:
    record_WAW.geometry = unary_union(waw_geoms)
db._records['1300000'] = record_WAW

# --- REGION MAZOWIECKI REGIONALNY ---
record_MAZ = gtdb.TERYTRecord(teryt_id="1500000",
                              name="Mazowiecki regionalny")
record_MAZ.level = 2
record_MAZ.kind = 0
record_MAZ.parent_id = '0000000'
mazowieckie_record = db._records.get('1200000')
if mazowieckie_record:
    all_maz_children = set(mazowieckie_record.children_ids)
    waw_children = set(record_WAW.children_ids)
    maz_children = sorted(all_maz_children - waw_children)
    record_MAZ.children_ids = maz_children
    maz_geoms = [db._records[tid].geometry for tid in maz_children
                 if tid in db._records and db._records[tid].geometry is not None]
    if maz_geoms:
        record_MAZ.geometry = unary_union(maz_geoms)
db._records['1500000'] = record_MAZ

print(f"Added REGION WARSZAWSKI STOŁECZNY (1300000): {len(record_WAW.children_ids)} children")
print(f"Added REGION MAZOWIECKI REGIONALNY (1500000): {len(record_MAZ.children_ids)} children")
print(f"\nTotal records in database: {len(db._records)}")

Added 49 old voivodeship records to database
Added REGION WARSZAWSKI STOŁECZNY (1300000): 10 children
Added REGION MAZOWIECKI REGIONALNY (1500000): 22 children

Total records in database: 4612


In [6]:
# =============================================================================
# STEP 3: Load BDL Source Data
# =============================================================================
df_demographic = pd.read_csv(gus_root / "data" / 'bdl_demographic_data.csv', encoding='utf-8')
df_educ_add_1 = pd.read_csv(gus_root / "data" / "P2350_BDL_educ_1995_2020 [remove data for 2000].csv",
                             encoding='utf-8', sep=';')
df_educ_add_2 = pd.read_csv(gus_root / "data" / "P4092_BDL_educ_2010_2024.csv",
                             encoding='utf-8', sep=';')
df_c_1988 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP1988_data.csv', encoding='utf-8')
df_c_2002 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2002_data.csv', encoding='utf-8')
df_c_2011 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2011_data.csv', encoding='utf-8')
df_c_2021 = pd.read_csv(gus_root / "data" / "census_data" / 'NSP2021_data.csv', encoding='utf-8')
df_c_2021_add = pd.read_csv(gus_root / "data" / "census_data" / 'P4315_Census_2021_educ_sex.csv',
                             encoding='utf-8', sep=';')


df_variables = pd.read_csv(gus_root / "metadata" / 'bdl_variables_level6.csv', encoding='utf-8')
df_c_variables = pd.read_csv(gus_root / "metadata" / 'census_meta.csv', encoding='utf-8')

print(f"df_demographic: {df_demographic.shape}")
print(f"df_educ_add_1: {df_educ_add_1.shape}")
print(f"df_educ_add_2: {df_educ_add_2.shape}")
print(f"df_variables: {df_variables.shape}")
print(f"df_c_2021_add: {df_c_2021_add.shape}")
print(f"  Columns: {list(df_c_2021_add.columns[:5])} ...")
print(f"\nAvailable subjects: {sorted(df_demographic['subjectId'].unique())}")

df_demographic: (431358, 5)
df_educ_add_1: (19, 133)
df_educ_add_2: (19, 78)
df_variables: (27922, 10)
df_c_2021_add: (4196, 33)
  Columns: ['Kod', 'Nazwa', 'ogółem;ogółem;2021;[osoba]', 'ogółem;wyższe;2021;[osoba]', 'ogółem;średnie i policealne - ogółem;2021;[osoba]'] ...

Available subjects: ['P1336', 'P2137', 'P2914']


In [7]:
# Edit years for census of 1988

for id, row in df_c_1988.iterrows():
    df_c_1988.at[id, 'values']= df_c_1988.at[id, 'values'].replace(", 'year': '1998'", ", 'year': '1988'")
    
df_c_variables['years'] = df_c_variables['years'].str.replace("[1998]", "[1988]")

# Unify the meta dataframe structure
df_c_variables['n4'] = None
df_c_variables['n5'] = None


In [8]:
# Collect all subject ids
# Subjects to ignore entirely (urban/rural splits, city-only data)
IGNORED_SUBJECTS = {'P4345', 'P3310', 'P1336', 'P2914'}

subject_ids = {"BDL": [], "Census": {"1988" : [], "2002": [], "2011": [], "2021": []}}

subject_ids["BDL"] = [s for s in df_demographic['subjectId'].unique() if s not in IGNORED_SUBJECTS]
subject_ids["Census"]["1988"] = [s for s in df_c_1988['subjectId'].unique() if s not in IGNORED_SUBJECTS]
subject_ids["Census"]["2002"] = [s for s in df_c_2002['subjectId'].unique() if s not in IGNORED_SUBJECTS]
subject_ids["Census"]["2011"] = [s for s in df_c_2011['subjectId'].unique() if s not in IGNORED_SUBJECTS]
subject_ids["Census"]["2021"] = [s for s in df_c_2021['subjectId'].unique() if s not in IGNORED_SUBJECTS]

subject_ids_flat = subject_ids['BDL'] + subject_ids["Census"]["1988"] + subject_ids["Census"]["2002"] \
    + subject_ids["Census"]["2011"] + subject_ids["Census"]["2021"]

subject_names_dict = {}
for subject in subject_ids_flat:
    subject_names_dict[subject] = ''
    
subject_ids

{'BDL': ['P2137'],
 'Census': {'1988': ['P2884', 'P2885', 'P2883', 'P2887'],
  '2002': ['P2114', 'P2403', 'P2402', 'P2871'],
  '2011': ['P3304', 'P3311', 'P3309', 'P3420'],
  '2021': ['P4253', 'P4320', 'P4287']}}

In [9]:
# Names of different subjects
# BDL subject
subject_names_dict['P2137'] = 'pop__age_sex'

# Census 1988
subject_names_dict['P2884'] = 'pop__age'
subject_names_dict['P2885'] = 'pop__educ'
subject_names_dict['P2883'] = 'pop__sex'
subject_names_dict['P2887'] = 'hh_size'

# Census 2002
subject_names_dict['P2114'] = 'pop__age_sex'
subject_names_dict['P2403'] = 'pop__age_educ'
subject_names_dict['P2402'] = 'pop__sex_educ'
subject_names_dict['P2871'] = 'hh_size'

# Census 2011
subject_names_dict['P3304'] = 'pop__age_sex'
subject_names_dict['P3311'] = 'pop__age_educ'
subject_names_dict['P3309'] = 'pop__sex_educ'
subject_names_dict['P3420'] = 'hh_size'

# Census 2021
subject_names_dict['P4253'] = 'pop__age_sex'
subject_names_dict['P4320'] = 'pop__age_educ'
subject_names_dict['P4315'] = 'pop__sex_educ'  # Wide-format CSV (added manually)
subject_names_dict['P4287'] = 'hh_size'

subject_names_dict

{'P2137': 'pop__age_sex',
 'P2884': 'pop__age',
 'P2885': 'pop__educ',
 'P2883': 'pop__sex',
 'P2887': 'hh_size',
 'P2114': 'pop__age_sex',
 'P2403': 'pop__age_educ',
 'P2402': 'pop__sex_educ',
 'P2871': 'hh_size',
 'P3304': 'pop__age_sex',
 'P3311': 'pop__age_educ',
 'P3309': 'pop__sex_educ',
 'P3420': 'hh_size',
 'P4253': 'pop__age_sex',
 'P4320': 'pop__age_educ',
 'P4287': 'hh_size',
 'P4315': 'pop__sex_educ'}

In [10]:
# =============================================================================
# STEP 4: Process Subjects
# =============================================================================
# Use the new process_subject_data() static method on GeoTERYTDatabase

# Example: Process subject P2137 (Population Data) from BDL
subject_id = 'P2137'
df_p2137 = gtdb.GeoTERYTDatabase.process_subject_data(df_demographic, df_variables, subject_id)

print(f"Processed P2137: {df_p2137.shape}")
print(f"\nColumns: {list(df_p2137.columns)}")
print(f"\nCategory columns present:")
for col in ['n1', 'n2', 'n3', 'n4', 'n5']:
    if col in df_p2137.columns:
        print(f"  {col}: {sorted(df_p2137[col].dropna().unique())}")
print(f"\nYears: {sorted(df_p2137['year'].dropna().astype(str).unique())}")
print(f"Unique TERYT IDs: {df_p2137['teryt_id'].nunique()}")
df_p2137.head()

Processed P2137: (7355712, 11)

Columns: ['nuts_id', 'name', 'variableId', 'subjectId', 'var_id', 'n1', 'n2', 'year', 'val', 'attrId', 'teryt_id']

Category columns present:
  n1: ['0-14', '0-4', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '5-9', '50-54', '55-59', '60-64', '65-69', '70 i więcej', '70-74', '75-79', '80-84', '85 i więcej', 'ogółem']
  n2: ['kobiety', 'mężczyźni', 'ogółem']

Years: ['1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Unique TERYT IDs: 4556


,nuts_id,name,variableId,subjectId,var_id,n1,n2,year,val,attrId,teryt_id
0,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1995,38609399,1,0000000
1,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1996,38639341,1,0000000
2,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1997,38659979,1,0000000
3,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1998,38666983,1,0000000
4,000000000000,POLSKA,72305,P2137,72305,ogółem,ogółem,1999,38263303,1,0000000


In [11]:
# =============================================================================
# STEP 4: Process ALL Subjects
# =============================================================================

df_subjects = {
    "BDL": df_demographic,
    "Census": {
        "1988": df_c_1988,
        "2002": df_c_2002,
        "2011": df_c_2011,
        "2021": df_c_2021
    }
}

df_processed_subjects = {
    "BDL": {},
    "Census": {
        "1988": {},
        "2002": {},
        "2011": {},
        "2021": {}
    }
}

# --- Process BDL and Census subjects ---
for subject in subject_ids.items():
    if subject[0] == "BDL":
        subjects = subject[1]
        for s in subjects:
            print(f"Processing BDL subject: {s}...")
            df = gtdb.GeoTERYTDatabase.process_subject_data(df_demographic, df_variables, s)
            df_processed_subjects["BDL"][s] = df
    else:
        for sub in subject[1].items():
            print(f"Processing Census subject: {sub[0]} - {sub[1]}...")
            year = sub[0]
            df_c = df_subjects["Census"][year]
            for s in sub[1]:
                print(f"  Processing subject: {s}...")
                df = gtdb.GeoTERYTDatabase.process_subject_data(df_c, df_c_variables, s)
                df_processed_subjects["Census"][year][s] = df

# --- P4315: Convert wide-format Census 2021 pop__sex_educ to long format ---
print("\nProcessing P4315 (wide-format Census 2021 pop__sex_educ)...")

rows = []
var_id_counter = 900001
var_id_lookup = {}  # (sex, educ) -> var_id

for col in df_c_2021_add.columns:
    if col in ['Kod', 'Nazwa']:
        continue
    parts = col.split(';')
    if len(parts) < 3:
        continue
    sex = parts[0].strip()
    educ = parts[1].strip()
    year = int(parts[2].strip())
    key = (sex, educ)
    if key not in var_id_lookup:
        var_id_lookup[key] = var_id_counter
        var_id_counter += 1
    vid = var_id_lookup[key]
    
    for _, row in df_c_2021_add.iterrows():
        kod = str(row['Kod']).zfill(7)
        val = row[col]
        if pd.isna(val):
            continue
        rows.append({
            'nuts_id': kod.ljust(12, '0'),
            'name': row['Nazwa'],
            'variableId': vid,
            'subjectId': 'P4315',
            'var_id': vid,
            'n1': sex,
            'n2': educ,
            'year': year,
            'val': float(val),
            'teryt_id': kod
        })

df_p4315 = pd.DataFrame(rows)
df_processed_subjects["Census"]["2021"]["P4315"] = df_p4315
# Also add to subject_ids so downstream loading picks it up
if 'P4315' not in subject_ids["Census"]["2021"]:
    subject_ids["Census"]["2021"].append('P4315')
print(f"  P4315: {df_p4315.shape[0]:,} rows, {df_p4315['teryt_id'].nunique()} units")
print(f"  Variables: {len(var_id_lookup)} (sex x education combos)")

# --- Fix hh_size subjects ---

# P3420 (Census 2011): Remove 'wskaźnik precyzji' rows, keep 'wartość liczbowa'.
# Remove 'ludność w gospodarstwach domowych' and 'przeciętna liczba osób'.
# Relabel to standard hh_size format for bin parsing compatibility.
# NOTE: verified that values ARE household counts (sum of bins = ogółem).
# NOTE: P3420 is powiat-level (11-digit IDs), not gmina-level.
if 'P3420' in df_processed_subjects["Census"]["2011"]:
    df = df_processed_subjects["Census"]["2011"]["P3420"]
    before = len(df)
    if 'n2' in df.columns:
        df = df[df['n2'] == 'wartość liczbowa'].copy()
        if df['n2'].nunique() <= 1:
            df = df.drop(columns=['n2'])
    if 'n1' in df.columns:
        exclude = ['ludność w gospodarstwach domowych',
                    'przeciętna liczba osób w gospodarstwie domowym']
        df = df[~df['n1'].isin(exclude)].copy()
        # Relabel to standard format for _parse_numeric_bounds compatibility
        label_map = {
            'osoby w gospodarstwie domowym - 1': '1-osobowe',
            'osoby w gospodarstwie domowym - 2': '2-osobowe',
            'osoby w gospodarstwie domowym - 3': '3-osobowe',
            'osoby w gospodarstwie domowym - 4': '4-osobowe',
            'osoby w gospodarstwie domowym - 5 i więcej': '5-osobowe i większe',
        }
        df['n1'] = df['n1'].replace(label_map)
    df_processed_subjects["Census"]["2011"]["P3420"] = df
    print(f"\n  P3420 fix: {before} -> {len(df)} rows (filtered + relabeled)")

# P2871 (Census 2002): Keep only 'gospodarstwa' rows (not 'ludność w gospodarstwach')
if 'P2871' in df_processed_subjects["Census"]["2002"]:
    df = df_processed_subjects["Census"]["2002"]["P2871"]
    before = len(df)
    if 'n1' in df.columns:
        df = df[df['n1'] == 'gospodarstwa'].copy()
        if df['n1'].nunique() <= 1:
            df = df.drop(columns=['n1'])
    df_processed_subjects["Census"]["2002"]["P2871"] = df
    print(f"  P2871 fix: {before} -> {len(df)} rows (kept only 'gospodarstwa')")

# P4287 (Census 2021): Remove 'ludność w gospodarstwach domowych' and 'przeciętna'
if 'P4287' in df_processed_subjects["Census"]["2021"]:
    df = df_processed_subjects["Census"]["2021"]["P4287"]
    before = len(df)
    if 'n1' in df.columns:
        exclude = ['ludność w gospodarstwach domowych',
                    'przeciętna liczba osób w gospodarstwie domowym']
        df = df[~df['n1'].isin(exclude)].copy()
    df_processed_subjects["Census"]["2021"]["P4287"] = df
    print(f"  P4287 fix: {before} -> {len(df)} rows (removed useless vars)")

del df_subjects
gc.collect()

# Save df_processed_subjects for later use
import pickle
with open(gus_root / 'data' / 'processed.pkl', 'wb') as f:
    pickle.dump(df_processed_subjects, f)

Processing BDL subject: P2137...
Processing Census subject: 1988 - ['P2884', 'P2885', 'P2883', 'P2887']...
  Processing subject: P2884...
  Processing subject: P2885...
  Processing subject: P2883...
  Processing subject: P2887...
Processing Census subject: 2002 - ['P2114', 'P2403', 'P2402', 'P2871']...
  Processing subject: P2114...
  Processing subject: P2403...
  Processing subject: P2402...
  Processing subject: P2871...
Processing Census subject: 2011 - ['P3304', 'P3311', 'P3309', 'P3420']...
  Processing subject: P3304...
  Processing subject: P3311...
  Processing subject: P3309...
  Processing subject: P3420...
Processing Census subject: 2021 - ['P4253', 'P4320', 'P4287']...
  Processing subject: P4253...
  Processing subject: P4320...
  Processing subject: P4287...

Processing P4315 (wide-format Census 2021 pop__sex_educ)...
  P4315: 125,880 rows, 4196 units
  Variables: 30 (sex x education combos)

  P3420 fix: 6064 -> 2274 rows (filtered + relabeled)
  P2871 fix: 43776 -> 21

In [12]:
# =============================================================================
# STEP 4C: Process Additional Education Data (P2350 & P4092)
# =============================================================================
# These BDL datasets are in wide format with semicolon-separated column names:
# "ludność ogółem wg BAEL;{educ_level};wartość liczbowa;{year};[tys. osób]"
# We flatten the two useless dimensions (constant labels) and convert to long format.
# Values are in thousands (tys. osób) → multiply by 1000.

def bdl_kod_to_teryt(kod):
    """Convert 12-digit BDL Kod to 7-digit TERYT ID."""
    kod_str = str(int(kod)).zfill(11)
    if kod_str == '000000000000':
        return '0000000'
    sub = kod_str[1:5]
    if sub == '1410':
        return '1300000'  # Warszawski stołeczny
    elif sub == '1420':
        return '1500000'  # Mazowiecki regionalny
    woj = kod_str[1:3]
    return woj + '00000'

def process_wide_educ_csv(df_wide, subject_id, set_nan_year=None):
    """
    Convert wide-format BDL education CSV to long format for load_subject_data().
    
    - Flattens useless dimensions ('ludność ogółem wg BAEL', 'wartość liczbowa')
    - Converts Kod to 7-digit TERYT ID
    - Multiplies values by 1000 (tys. osób → persons)
    - Replaces year set_nan_year with NaN (if specified)
    - Replaces 0 with NaN
    - Linearly interpolates NaN values within each (teryt, education) series
    """
    rows = []
    var_id_counter = 800001
    var_id_lookup = {}
    
    for col in df_wide.columns:
        if col in ['Kod', 'Nazwa']:
            continue
        if pd.isna(col) or str(col).strip() == '':
            continue
        
        parts = str(col).split(';')
        if len(parts) < 4:
            continue
        
        # parts[0] = "ludność ogółem wg BAEL" (flatten away)
        educ = parts[1].strip()
        # parts[2] = "wartość liczbowa" (flatten away)
        year = int(parts[3].strip())
        
        if educ not in var_id_lookup:
            var_id_lookup[educ] = var_id_counter
            var_id_counter += 1
        vid = var_id_lookup[educ]
        
        for _, row in df_wide.iterrows():
            teryt_id = bdl_kod_to_teryt(row['Kod'])
            val = row[col]
            if pd.isna(val):
                continue
            
            # Convert from thousands to actual count
            val = float(val) * 1000
            
            # Replace year with NaN if requested
            if set_nan_year and year == set_nan_year:
                val = np.nan
            
            # Replace 0 with NaN
            if not pd.isna(val) and val == 0:
                val = np.nan
            
            rows.append({
                'teryt_id': teryt_id,
                'name': row['Nazwa'],
                'variableId': vid,
                'subjectId': subject_id,
                'var_id': vid,
                'n1': educ,
                'year': year,
                'val': val
            })
    
    df_long = pd.DataFrame(rows)
    
    # Linear interpolation: for each (teryt_id, educ_level), interpolate NaN values
    interpolated_rows = []
    for (tid, educ), grp in df_long.groupby(['teryt_id', 'n1']):
        grp_sorted = grp.sort_values('year')
        series = grp_sorted.set_index('year')['val']
        series = series.interpolate(method='linear')
        
        for yr, v in series.items():
            if pd.isna(v):
                continue
            ref_row = grp_sorted[grp_sorted['year'] == yr].iloc[0]
            interpolated_rows.append({
                'teryt_id': tid,
                'name': ref_row['name'],
                'variableId': ref_row['variableId'],
                'subjectId': subject_id,
                'var_id': ref_row['var_id'],
                'n1': educ,
                'year': yr,
                'val': v
            })
    
    return pd.DataFrame(interpolated_rows)


# Process P2350 (1995-2020, year 2000 = NaN)
print("Processing P2350 (BDL education 1995-2020)...")
df_p2350 = process_wide_educ_csv(df_educ_add_1, 'P2350', set_nan_year=2000)
print(f"  P2350: {len(df_p2350):,} rows, {df_p2350['teryt_id'].nunique()} units")

# Process P4092 (2010-2024)
print("Processing P4092 (BDL education 2010-2024)...")
df_p4092 = process_wide_educ_csv(df_educ_add_2, 'P4092')
print(f"  P4092: {len(df_p4092):,} rows, {df_p4092['teryt_id'].nunique()} units")

# Add to processed subjects
df_processed_subjects["BDL"]["P2350"] = df_p2350
df_processed_subjects["BDL"]["P4092"] = df_p4092

# Add to subject IDs and names
if 'P2350' not in subject_ids["BDL"]:
    subject_ids["BDL"].append('P2350')
if 'P4092' not in subject_ids["BDL"]:
    subject_ids["BDL"].append('P4092')

subject_names_dict['P2350'] = 'pop__educ'
subject_names_dict['P4092'] = 'pop__educ'

print(f"\n  Education categories (P2350): {sorted(df_p2350['n1'].unique())}")
print(f"  Education categories (P4092): {sorted(df_p4092['n1'].unique())}")
print(f"  Years P2350: {sorted(df_p2350['year'].unique())}")
print(f"  Years P4092: {sorted(df_p4092['year'].unique())}")

# Update the processed.pkl with the new data
import pickle
with open(gus_root / 'data' / 'processed.pkl', 'wb') as f:
    pickle.dump(df_processed_subjects, f)
print("\n  ✓ Updated processed.pkl with P2350 and P4092")

Processing P2350 (BDL education 1995-2020)...
  P2350: 2,410 rows, 19 units
Processing P4092 (BDL education 2010-2024)...
  P4092: 1,425 rows, 19 units

  Education categories (P2350): ['gimnazjalne, podstawowe i niższe', 'policealne oraz średnie zawodowe/branżowe', 'wyższe', 'zasadnicze zawodowe/branżowe', 'średnie ogólnokształcące']
  Education categories (P4092): ['gimnazjalne, podstawowe i niższe', 'policealne oraz średnie zawodowe/branżowe', 'wyższe', 'zasadnicze zawodowe/branżowe', 'średnie ogólnokształcące']
  Years P2350: [np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
  Years P4092: [np.int64(2010), np.int64(2011),

In [13]:
# Open the saved processed data to verify
import pickle
with open(gus_root / 'data' / 'processed.pkl', 'rb') as f:
    df_processed_subjects = pickle.load(f)
    

In [14]:
# =============================================================================
# STEP 5: Load Subject Data onto TERYTRecords
# =============================================================================
# This attaches time series data to each matching TERYTRecord in the database

for subject in subject_ids.items():
    if subject[0] == "BDL":
        subjects = subject[1]
        for s in subjects:
            print(f"Loading BDL subject: {s}...")
            df = df_processed_subjects["BDL"][s]
            stats = db.load_subject_data(df, source_type='BDL', subject_id=s, subject_name=subject_names_dict[s])
            print(f"  Loading statistics: {stats['matched_teryts']} matched, {stats['unmatched_teryts']} unmatched")
            print(f"  Total data points loaded: {stats['total_data_points']:,}")
    else:
        for sub in subject[1].items():
            print(f"Loading Census subject: {sub[0]} - {sub[1]}...")
            year = sub[0]
            for s in sub[1]:
                print(f"  Loading subject: {s}...")
                df = df_processed_subjects["Census"][year][s]
                stats = db.load_subject_data(df, source_type='Census', subject_id=s, subject_name=subject_names_dict[s])
                print(f"    Loading statistics: {stats['matched_teryts']} matched, {stats['unmatched_teryts']} unmatched")
                print(f"    Total data points loaded: {stats['total_data_points']:,}")

# Free memory
del df_processed_subjects
gc.collect()

Loading BDL subject: P2137...
  ✓ Loaded 7,352,352 data points for subject P2137
  ✓ Matched 4549 TERYT records, 7 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0216001', '0410001', '1210001', '1431981', '1431991', '1465158', '1465998']
  Loading statistics: 4549 matched, 7 unmatched
  Total data points loaded: 7,352,352
Loading BDL subject: P2350...
  ✓ Loaded 2,410 data points for subject P2350
  ✓ Matched 19 TERYT records, 0 unmatched
  Loading statistics: 19 matched, 0 unmatched
  Total data points loaded: 2,410
Loading BDL subject: P4092...
  ✓ Loaded 1,425 data points for subject P4092
  ✓ Matched 19 TERYT records, 0 unmatched
  Loading statistics: 19 matched, 0 unmatched
  Total data points loaded: 1,425
Loading Census subject: 1988 - ['P2884', 'P2885', 'P2883', 'P2887']...
  Loading subject: P2884...
  ✓ Loaded 28,992 data points for subject P2884
  ✓ Matched 3624 TERYT records, 5 unmatched
  ⚠ Unmatched TERYT IDs (first 10): ['0216001', '0410001', '1210001', '1431981', '1431

0

In [15]:
# =============================================================================
# STEP 6: Create Merged Subjects (unified census + BDL time series)
# =============================================================================
# Creates NEW merged subjects (M_ prefix) from groups sharing the same topic.
# Original subjects are NOT modified - raw data stays intact.
# For age dimensions: computes unified bins via common break points.
# For sex dimensions: exact label matching.

importlib.reload(gtdb)

print("Creating merged subjects...")
merged_info = db.create_merged_subjects(subject_names_dict)

# Update subject_names_dict with merged subjects
for merged_sid, source_ids in merged_info.items():
    group_name = merged_sid.replace('M_', '')
    subject_names_dict[merged_sid] = group_name
    print(f"  Added {merged_sid} -> '{group_name}'")

# Show data summary after merge
summary = db.get_data_summary()
print(f"\nAfter merge:")
print(f"  Records with data: {summary['records_with_data']}")
print(f"  Subjects: {summary['n_subjects']} ({summary['subjects']})")
print(f"  Total data series: {summary['total_data_series']:,}")
print(f"  Total data points: {summary['total_data_points']:,}")

Creating merged subjects...
Found 5 subject groups to merge:
  pop__age_sex: ['P2137', 'P2114', 'P3304', 'P4253']
  pop__educ: ['P2885', 'P2350', 'P4092']
  hh_size: ['P2887', 'P2871', 'P3420', 'P4287']
  pop__age_educ: ['P2403', 'P3311', 'P4320']
  pop__sex_educ: ['P2402', 'P3309', 'P4315']

  Merged subject: M_pop__age_sex
    n1 (['age']): 19 labels
    n2 (['sex']): 3 labels
    Aggregates detected in P2137 dim n1: {'70 i więcej', '0-14'}
    ✓ 4533 records, 257445 merged series created

  Merged subject: M_pop__educ
    n1 (['education']): 8 labels
    ✓ 3643 records, 14591 merged series created

  Merged subject: M_hh_size
    n1 (['age']): 5 labels
    ✓ 3624 records, 14496 merged series created

  Merged subject: M_pop__age_educ
    n1 (['age']): 11 labels
    n2 (['education']): 17 labels
    ✓ 381 records, 71060 merged series created

  Merged subject: M_pop__sex_educ
    n1 (['sex']): 3 labels
    n2 (['education']): 16 labels
    ✓ 4275 records, 193113 merged series created

In [16]:
# =============================================================================
# STEP 7: Extract Total Population & Classify Urban/Rural
# =============================================================================

print("Extracting total population...")
n_pop = db.extract_population(subject_names_dict)

print("\nClassifying urban/rural...")
n_class = db.classify_population()

Extracting total population...
  ✓ Extracted population for 4533 records

Classifying urban/rural...
  ✓ Classified 3411 records by urban/rural


In [17]:
# =============================================================================
# STEP 8: Code Dimension Labels
# =============================================================================

print("Coding dimension labels...")
n_coded = db.code_dimension_labels(subject_names_dict)

Coding dimension labels...
  ✓ Coded dimension labels for 1953484 DataSeries across 24 subjects


In [18]:
# =============================================================================
# STEP 9: Save Database with All Data
# =============================================================================

save_path = geo_root / 'geoteryt_complete_final.pkl'
db.save_complete(save_path)

Saving complete database to /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl...
  ✓ Saved 4612 records
  ✓ Records with data: 4535
  ✓ File size: 1585.8 MB
  ✓ Path: /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/geoteryt_complete_final.pkl


In [19]:
# =============================================================================
# STEP 10: Verify Data on Individual Records
# =============================================================================
data_summary = db.get_data_summary()
print("Data Summary:")
for k, v in data_summary.items():
    print(f"  {k}: {v}")

# Show subjects and their types
print("\nSubjects:")
for sid in sorted(data_summary['subjects']):
    sname = subject_names_dict.get(sid, '')
    prefix = "MERGED" if sid.startswith('M_') else "RAW"
    print(f"  [{prefix}] {sid}: {sname}")

Data Summary:
  records_with_data: 4535
  total_records: 4612
  subjects: ['M_hh_size', 'M_pop__age_educ', 'M_pop__age_sex', 'M_pop__educ', 'M_pop__sex_educ', 'P2114', 'P2137', 'P2350', 'P2402', 'P2403', 'P2871', 'P2883', 'P2884', 'P2885', 'P2887', 'P3304', 'P3309', 'P3311', 'P3420', 'P4092', 'P4253', 'P4287', 'P4315', 'P4320']
  n_subjects: 24
  total_data_series: 1953484
  total_data_points: 15331778

Subjects:
  [MERGED] M_hh_size: hh_size
  [MERGED] M_pop__age_educ: pop__age_educ
  [MERGED] M_pop__age_sex: pop__age_sex
  [MERGED] M_pop__educ: pop__educ
  [MERGED] M_pop__sex_educ: pop__sex_educ
  [RAW] P2114: pop__age_sex
  [RAW] P2137: pop__age_sex
  [RAW] P2350: pop__educ
  [RAW] P2402: pop__sex_educ
  [RAW] P2403: pop__age_educ
  [RAW] P2871: hh_size
  [RAW] P2883: pop__sex
  [RAW] P2884: pop__age
  [RAW] P2885: pop__educ
  [RAW] P2887: hh_size
  [RAW] P3304: pop__age_sex
  [RAW] P3309: pop__sex_educ
  [RAW] P3311: pop__age_educ
  [RAW] P3420: hh_size
  [RAW] P4092: pop__educ
  [